# Getting Started with Lichess DB

This demo is intended to guide your first interaction with the Lichess database.

## Table of Contents

1. [Setup](#Setup)
    - [Importing dependencies](#Importing-dependencies)
2. [Connecting to DB](#Connecting-to-DB)
3. [SQL Examples](#SQL-Examples)
    - [Check all existing tables](#Check-all-existing-tables)
    - [View table schema](#View-table-schema)
    - [Query players table](#Query-players-table)
    - [Query games table](#Query-games-table)
    - [Query moves table](#Query-moves-table)
4. [Remove tables](#Remove-tables)
5. [Closing DB Connection](#Closing-DB-Connection)
6. [Tips](#Tips)

## Setup

### Importing dependencies

Let's first import packages to use.

In [1]:
import duckdb
import pandas as pd

Note that `pandas` is imported here to turn query results into `DataFrame`s, which are easier to work with for downstream analytic tasks. In other words, it is for convenience, not necessity.

On the other hand, `duckdb` is a necessary package for connecting and querying the database.

## Connecting to DB

DuckDB supports multiple concurrent connections for read but NOT for write. We hence restrict database write access to the designated person(s) only, who are responsible for maintaining and updating the database.

Other users of the database have two options:
1. Connect to the database in the read-only mode
2. Create a personal database and “attach” the primary database in the read-only mode

The advantage of the second option is that the user can not only read from the primary database ("source of truth") but also create and store new tables in their personal database ("handy place"). This way, each individual researcher has full control over their analytic activities.

Let's try the second option to connect to the database.

In [7]:
# Connect to the personal database
conn = duckdb.connect(
    database="/scratch/gpfs/GRIFFITHS/hl4291/personal.db",
    config={
        "threads": 10,
        "memory_limit": "60GB",
        "temp_directory": ".",
    }
)

The above code establishes connection to a personal database named `personal.db`. Note that it automatically creates a new database file if the specified one does not exist. Otherwise, it connects to the existing one.

`config` above specifies the resources we are giving to the current connection. In this case, we are allowing database operations (e.g., queries) to use up to 10 CPUs and 60GB of memory.

We should always explicitly specify the amount of resources that WE are allowed to use. For instance, we here specify 10 CPUs and 60GB of memory because the notebook session was created with at least this amount of resources.

If these limits are not explicitly specified, the database will try to use as many resources as possible, which can cause problems in shared computing environments like Della.

In DuckDB, larger-than-memory workloads are supported by spilling to disk. That is, it will create and use temporary files when the available memory is no longer sufficient to continue processing. `temp_directory` above specifies the location to store these temporary files, and it is important to ensure this location has sufficient storage.

Next, let's "attach" the primary database to this personal one so that we can read data from the former.

In [ ]:
# Attach the primary database
conn.sql("""ATTACH '/scratch/gpfs/GRIFFITHS/chess-db/lichess.db' AS core (READ_ONLY);""")

The above code attaches a database named `lichess.db` to our current database (i.e., `personal.db`) in read-only mode. Note that omitting the `READ_ONLY` specification will result in an error if you do not have write permission for the attached database.

Aliased as `core`, the primary database can be queried as follows:

In [9]:
# Query `games` table in the primary database
conn.sql("""
SELECT
    *
FROM
    core.games
LIMIT 5;
""")

┌─────────────────┬──────────┬─────────────────────┬───────────────┬─────────────────┬─────────────────────────────────────────────┬──────────────────┬───────────────────┬────────────────────────────────────────────────────────────────────────────────────┬────────────────────────────────────────────────────────────────────────────────────┬───────────┬───────────┬──────────────────┬──────────────────┬───────────┬───────────┬─────────────────────────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────┬─────────┬──────────────────────────┬──────────────────────────┬───────────────┬────────────────────────────┬──────────────────────┬─────────────────────┬─────────────────────┬─────────────────────────┬───────────────┬────────────────────────────┬──────────────────────┬─────────────────────┬─────────────────────┬─────────────────────────┬───────────────────────┬───────────────────────┬──────────────┬──────────────┐
│       g

With `.df()`, we can turn the query result into a Pandas `DataFrame`, like so:

In [10]:
# Query `games` table in the primary database
conn.sql("""
SELECT
    *
FROM
    core.games
LIMIT 5;
""").df()

,gid,id,utc_datetime,initial_clock,clock_increment,opening,white_id,black_id,white_title,black_title,...,black_n_games,black_n_games_ultra_bullet,black_n_games_bullet,black_n_games_blitz,black_n_games_rapid,black_n_games_classical,white_time_since_last,black_time_since_last,white_tenure,black_tenure
0,201301001000000,j1dkb5dw,2012-12-31 23:01:03,600,8,French Defense: Normal Variation,BFG9k,mamalak,NaN,NaN,...,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,0
1,201301001000001,a9tcp02g,2012-12-31 23:04:12,480,2,"Queen's Pawn Game: Colle System, Anti-Colle",Desmond_Wilson,savinka59,NaN,NaN,...,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,0
2,201301001000002,szom2tog,2012-12-31 23:03:15,420,17,Four Knights Game: Italian Variation,Kozakmamay007,VanillaShamanilla,NaN,NaN,...,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,0
3,201301001000003,rklpc7mk,2012-12-31 23:04:57,60,1,Caro-Kann Defense: Goldman Variation,Naitero_Nagasaki,800,NaN,NaN,...,0,<NA>,<NA>,<NA>,<NA>,<NA>,140,<NA>,0,0
4,201301001000004,1xb3os63,2012-12-31 23:02:37,60,1,French Defense: La Bourdonnais Variation,nichiren1967,Naitero_Nagasaki,NaN,NaN,...,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,0


## SQL Examples

The following examples demonstrate basic SQL statements to help you get familiar with the database. For a complete review of SQL support in DuckDB, please reference the [relevant documentation](https://duckdb.org/docs/sql/introduction).

### Check all existing tables

In [11]:
# Check all existing tables
conn.sql("""
SHOW ALL TABLES;
""").df()

,database,schema,name,column_names,column_types,temporary
0,core,main,games,"[gid, id, utc_datetime, initial_clock, clock_i...","[UBIGINT, VARCHAR, TIMESTAMP_S, USMALLINT, UTI...",False
1,core,main,moves,"[partition, gid, move_ply, move_uci, move_time...","[UINTEGER, UBIGINT, USMALLINT, VARCHAR, INTEGE...",False
2,core,main,players,"[id, first_game_datetime, last_game_datetime, ...","[VARCHAR, TIMESTAMP_S, TIMESTAMP_S, ENUM('GM',...",False


Since we have not yet created anything in our personal database, we only see tables in the primary database (aliased as `core`) and their details.

### View table schema

In [12]:
# View schema for the `games` table
conn.sql("""
DESCRIBE core.games;
""").df()

,column_name,column_type,null,key,default,extra
0,gid,UBIGINT,NO,None,None,None
1,id,VARCHAR,NO,None,None,None
2,utc_datetime,TIMESTAMP_S,YES,None,None,None
3,initial_clock,USMALLINT,YES,None,None,None
4,clock_increment,UTINYINT,YES,None,None,None
5,opening,VARCHAR,YES,None,None,None
6,white_id,VARCHAR,NO,None,None,None
7,black_id,VARCHAR,NO,None,None,None
8,white_title,"ENUM('GM', 'IM', 'FM', 'CM', 'NM', 'WGM', 'WIM...",YES,None,None,None
9,black_title,"ENUM('GM', 'IM', 'FM', 'CM', 'NM', 'WGM', 'WIM...",YES,None,None,None


### Query players table

The `players` table is relatively small, allowing for complex queries without significant concerns about compute and memory resources.

In [13]:
# Preview `players` table
conn.sql("""
SELECT
    *
FROM
    core.players
LIMIT 5;
""").df()

,id,first_game_datetime,last_game_datetime,title,tenure,n_consecutive_wins,elo,elo_ultra_bullet,elo_bullet,elo_blitz,elo_rapid,elo_classical,n_games,n_games_ultra_bullet,n_games_bullet,n_games_blitz,n_games_rapid,n_games_classical
0,BFG9k,2012-12-31 23:01:03,2024-05-31 20:48:50,NaN,4169,0,1574,<NA>,<NA>,1574,1679,<NA>,7531,0,0,6773,758,0
1,mamalak,2012-12-31 23:01:03,2013-08-15 17:02:22,NaN,227,1,1484,<NA>,<NA>,1356,1535,1484,223,0,0,2,50,171
2,arion_6,2012-12-31 23:02:14,2012-12-31 23:26:10,NaN,0,1,1799,<NA>,<NA>,1500,1799,<NA>,3,0,0,1,2,0
3,tiggran,2012-12-31 23:02:14,2020-04-30 00:02:36,NaN,2677,6,1707,1402,1776,1760,1814,1707,2580,27,1248,796,464,45
4,Naitero_Nagasaki,2012-12-31 23:02:37,2013-07-12 07:55:35,NaN,193,0,1581,<NA>,1871,1581,1747,<NA>,1287,0,746,477,64,0


In [14]:
# Count all players
conn.sql("""
SELECT
    COUNT(*)
FROM
    core.players;
""").df()

,count_star()
0,18049300


In [15]:
# Calculate tenure stats
conn.sql("""
SELECT
    MIN(tenure),
    MEDIAN(tenure),
    AVG(tenure),
    MAX(tenure)
FROM
    core.players;
""").df()

,min(tenure),median(tenure),avg(tenure),max(tenure)
0,0,30.0,281.053616,4169


In [16]:
# Analyze the number of games played per player
conn.sql("""
SELECT
    MIN(n_games),
    MEDIAN(n_games),
    AVG(n_games),
    MAX(n_games)
FROM
    core.players;
""").df()

,min(n_games),median(n_games),avg(n_games),max(n_games)
0,1,18.0,626.902036,803206


Let's say we are interested in players with the highest recent ratings.

In [17]:
# Identify 10 players with the highest recent ratings
conn.sql("""
SELECT
    *
FROM
    core.players
ORDER BY
    elo DESC
LIMIT 10;
""").df()

,id,first_game_datetime,last_game_datetime,title,tenure,n_consecutive_wins,elo,elo_ultra_bullet,elo_bullet,elo_blitz,elo_rapid,elo_classical,n_games,n_games_ultra_bullet,n_games_bullet,n_games_blitz,n_games_rapid,n_games_classical
0,I4000elo_suubscribe,2023-12-30 11:16:28,2023-12-30 11:52:46,NaN,0,0,3995,<NA>,3995,<NA>,<NA>,<NA>,33,0,33,0,0,0
1,PastaDoctorSsempa,2020-09-24 00:15:51,2020-09-24 00:51:24,NaN,0,19,3958,<NA>,3958,<NA>,<NA>,<NA>,19,0,19,0,0,0
2,wrd-Saif,2023-04-16 14:07:20,2023-04-16 15:55:46,NaN,0,19,3902,<NA>,3902,<NA>,<NA>,<NA>,63,0,63,0,0,0
3,Zakamura,2021-02-14 18:38:24,2021-02-14 18:46:28,NaN,0,0,3879,<NA>,3879,<NA>,<NA>,<NA>,26,0,26,0,0,0
4,e6f5b6,2020-09-22 21:01:05,2020-09-22 21:21:02,NaN,0,5,3845,<NA>,3845,<NA>,<NA>,<NA>,5,0,5,0,0,0
5,Jesjsssss,2024-04-13 05:40:58,2024-04-13 05:52:28,NaN,0,40,3773,<NA>,3773,<NA>,<NA>,<NA>,40,0,40,0,0,0
6,HooooHoooo,2016-01-18 21:53:05,2018-10-30 04:10:15,NaN,1016,57,3762,3762,2627,<NA>,<NA>,<NA>,240,21,219,0,0,0
7,Nasvfg,2020-09-22 20:42:01,2020-09-22 21:06:12,NaN,0,0,3602,<NA>,3602,<NA>,<NA>,<NA>,4,0,4,0,0,0
8,tothemoon1234,2022-03-14 16:18:10,2022-03-14 22:04:18,NaN,0,14,3594,<NA>,1958,3594,<NA>,<NA>,25,0,11,14,0,0
9,RandomEngine-AI,2023-07-28 18:49:45,2023-07-29 07:20:52,BOT,1,0,3574,<NA>,3574,<NA>,<NA>,<NA>,51,0,51,0,0,0


For later queries, let's store this result as a table in our personal database:

In [18]:
# Store query result as a table
conn.sql("""
CREATE OR REPLACE TABLE top_players AS (
    SELECT
        *
    FROM
        core.players
    ORDER BY
        elo DESC
    LIMIT 10
);
""")

In [19]:
# Check all existing tables
conn.sql("""
SHOW ALL TABLES;
""").df()

,database,schema,name,column_names,column_types,temporary
0,core,main,games,"[gid, id, utc_datetime, initial_clock, clock_i...","[UBIGINT, VARCHAR, TIMESTAMP_S, USMALLINT, UTI...",False
1,core,main,moves,"[partition, gid, move_ply, move_uci, move_time...","[UINTEGER, UBIGINT, USMALLINT, VARCHAR, INTEGE...",False
2,core,main,players,"[id, first_game_datetime, last_game_datetime, ...","[VARCHAR, TIMESTAMP_S, TIMESTAMP_S, ENUM('GM',...",False
3,personal,main,top_players,"[id, first_game_datetime, last_game_datetime, ...","[VARCHAR, TIMESTAMP_S, TIMESTAMP_S, ENUM('GM',...",False


### Query games table

The `games` table contains billions of records, so memory-intensive operations like table joins may not work well. However, we can first filter a desired subset and then perform complex queries on it.

In [20]:
# Preview `games` table
conn.sql("""
SELECT
    *
FROM
    core.games
LIMIT 5;
""").df()

,gid,id,utc_datetime,initial_clock,clock_increment,opening,white_id,black_id,white_title,black_title,...,black_n_games,black_n_games_ultra_bullet,black_n_games_bullet,black_n_games_blitz,black_n_games_rapid,black_n_games_classical,white_time_since_last,black_time_since_last,white_tenure,black_tenure
0,201301001000000,j1dkb5dw,2012-12-31 23:01:03,600,8,French Defense: Normal Variation,BFG9k,mamalak,NaN,NaN,...,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,0
1,201301001000001,a9tcp02g,2012-12-31 23:04:12,480,2,"Queen's Pawn Game: Colle System, Anti-Colle",Desmond_Wilson,savinka59,NaN,NaN,...,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,0
2,201301001000002,szom2tog,2012-12-31 23:03:15,420,17,Four Knights Game: Italian Variation,Kozakmamay007,VanillaShamanilla,NaN,NaN,...,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,0
3,201301001000003,rklpc7mk,2012-12-31 23:04:57,60,1,Caro-Kann Defense: Goldman Variation,Naitero_Nagasaki,800,NaN,NaN,...,0,<NA>,<NA>,<NA>,<NA>,<NA>,140,<NA>,0,0
4,201301001000004,1xb3os63,2012-12-31 23:02:37,60,1,French Defense: La Bourdonnais Variation,nichiren1967,Naitero_Nagasaki,NaN,NaN,...,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,0


In [21]:
# Count all games
conn.sql("""
SELECT
    COUNT(*)
FROM
    core.games;
""").df()

,count_star()
0,5657571460


In [22]:
# Count each game type
conn.sql("""
SELECT
    time_control_type,
    COUNT(*) AS frequency
FROM
    core.games
GROUP BY
    time_control_type
ORDER BY
    frequency DESC;
""").df()

,time_control_type,frequency
0,Blitz,2712237517
1,Bullet,1964281641
2,Rapid,826786783
3,UltraBullet,71865553
4,Classical,70277039
5,NaN,12122927


In [23]:
# Identify most frequent game openings
conn.sql("""
SELECT
    opening,
    COUNT(*) AS frequency
FROM
    core.games
GROUP BY
    opening
ORDER BY
    frequency DESC
LIMIT 10;
""").df()

,opening,frequency
0,Modern Defense,125129928
1,Queen's Pawn Game,117633120
2,Scandinavian Defense: Mieses-Kotroc Variation,110244043
3,Van't Kruijs Opening,105809507
4,Caro-Kann Defense,98876042
5,Philidor Defense,85227192
6,Horwitz Defense,80759322
7,French Defense: Knight Variation,77224818
8,Scandinavian Defense,76115134
9,Pirc Defense,64722576


Now, say we are interested in games that the recent best player has played. Let's identify these games using the information we saved earlier.

In [24]:
# Retrieve the recent best player
conn.sql("""
SELECT
    *
FROM
    top_players
LIMIT 1;
""").df()

,id,first_game_datetime,last_game_datetime,title,tenure,n_consecutive_wins,elo,elo_ultra_bullet,elo_bullet,elo_blitz,elo_rapid,elo_classical,n_games,n_games_ultra_bullet,n_games_bullet,n_games_blitz,n_games_rapid,n_games_classical
0,I4000elo_suubscribe,2023-12-30 11:16:28,2023-12-30 11:52:46,NaN,0,0,3995,<NA>,3995,<NA>,<NA>,<NA>,33,0,33,0,0,0


In [25]:
# Identify games by the recent best player
conn.sql("""
SELECT
    *
FROM
    core.games
WHERE
    utc_datetime BETWEEN '2023-12-30' AND '2023-12-31'
    AND (white_id = 'I4000elo_suubscribe' OR black_id = 'I4000elo_suubscribe');
""").df()

,gid,id,utc_datetime,initial_clock,clock_increment,opening,white_id,black_id,white_title,black_title,...,black_n_games,black_n_games_ultra_bullet,black_n_games_bullet,black_n_games_blitz,black_n_games_rapid,black_n_games_classical,white_time_since_last,black_time_since_last,white_tenure,black_tenure
0,202312092653864,INJwS6Rx,2023-12-30 11:16:28,60,0,French Defense,I4000elo_subscribe,I4000elo_suubscribe,NaN,NaN,...,0,<NA>,<NA>,<NA>,<NA>,<NA>,417,<NA>,0,0
1,202312092654950,oeoDlS6G,2023-12-30 11:17:00,60,0,French Defense,I4000elo_suubscribe,I4000elo_subscribe,NaN,NaN,...,46,0,46,0,0,0,32,32,0,0
2,202312092656331,ASqENB9a,2023-12-30 11:17:36,60,0,Van't Kruijs Opening,I4000elo_subscribe,I4000elo_suubscribe,NaN,NaN,...,2,0,2,0,0,0,36,36,0,0
3,202312092658817,922U9COI,2023-12-30 11:18:38,60,0,Van't Kruijs Opening,I4000elo_subscribee,I4000elo_suubscribe,NaN,NaN,...,3,0,3,0,0,0,547,62,0,0
4,202312092660410,xWMDh98J,2023-12-30 11:19:09,60,0,French Defense: Henneberger Variation,I4000elo_suubscribe,I4000elo_subscribee,NaN,NaN,...,16,0,16,0,0,0,31,31,0,0
5,202312092661613,ORTDoJzW,2023-12-30 11:19:45,60,0,Van't Kruijs Opening,I4000elo_subscribee,I4000elo_suubscribe,NaN,NaN,...,5,0,5,0,0,0,36,36,0,0
6,202312092665053,szwrIOSl,2023-12-30 11:21:18,60,0,Rat Defense: Small Center Defense,I4000elo_suubscribe,I4000elo_subscribee,NaN,NaN,...,18,0,18,0,0,0,93,93,0,0
7,202312092667821,kfWSMbU6,2023-12-30 11:22:18,60,0,French Defense,I4000elo_suubscribe,I4000elo_subscribee,NaN,NaN,...,19,0,19,0,0,0,60,60,0,0
8,202312092670617,XsQm31Dt,2023-12-30 11:23:28,60,0,Mieses Opening,I4000elo_suubscribe,I4000elo_subscribee,NaN,NaN,...,20,0,20,0,0,0,70,70,0,0
9,202312092672131,L0HLUNUe,2023-12-30 11:24:01,60,0,Mieses Opening,I4000elo_subscribee,I4000elo_suubscribe,NaN,NaN,...,9,0,9,0,0,0,33,33,0,0


Note that the query above contains a seemingly unnecessary additional condition, i.e., `utc_datetime BETWEEN '2023-12-30' AND '2023-12-31'`. However, this additional information helps to narrow down and speed up the search, which is critical in this case because the other queried columns (i.e., `white_id` and `black_id`) each contain numerous unique strings that are unsorted.

In general, it is advised that queries on the `games` table leverage filtering conditions that narrow down the target operation space.

Let's store this result too for later queries.

In [26]:
# Store query result as a table
conn.sql("""
CREATE OR REPLACE TABLE games_by_best_player AS (
    SELECT
        *
    FROM
        core.games
    WHERE
        utc_datetime BETWEEN '2023-12-30' AND '2023-12-31'
        AND (white_id = 'I4000elo_suubscribe' OR black_id = 'I4000elo_suubscribe')
);
""")

### Query moves table

The `moves` table contains hundreds of billions of records, making it impractical to run complex queries on the entire dataset. Hence, the recommendation is always to filter a desired subset first and then perform queries on it.

Eventually, we are interested in analyzing moves this best player makes in their games, for which we need to filter moves from all games that the player has played.

Given the scale of the `moves` table, we cannot resort to complex operations like table joins. Instead, we should query moves for each game at a time.

For example, we can retrieve moves from the best player's first game as follows:

In [27]:
# Get the first game by the best player
conn.sql("""
SELECT
    *
FROM
    games_by_best_player
ORDER BY
    utc_datetime
LIMIT 1;
""").df()

,gid,id,utc_datetime,initial_clock,clock_increment,opening,white_id,black_id,white_title,black_title,...,black_n_games,black_n_games_ultra_bullet,black_n_games_bullet,black_n_games_blitz,black_n_games_rapid,black_n_games_classical,white_time_since_last,black_time_since_last,white_tenure,black_tenure
0,202312092653864,INJwS6Rx,2023-12-30 11:16:28,60,0,French Defense,I4000elo_subscribe,I4000elo_suubscribe,NaN,NaN,...,0,<NA>,<NA>,<NA>,<NA>,<NA>,417,<NA>,0,0


In [28]:
# Retrieve moves from the first game by the best player
conn.sql("""
SELECT
    *
FROM
    core.moves
WHERE
    partition = 202312
    AND gid = 202312092653864;
""").df()

,partition,gid,move_ply,move_uci,move_time,player_white,player_clock_time,opponent_clock_time,board_position,castling_rights,en_passant_targets,halfmove_clock,n_possible_moves,n_pieces,player_in_check
0,202312,202312092653864,1,e2e4,<NA>,True,60,60,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR,KQkq,None,0,20,32,False
1,202312,202312092653864,2,e7e6,<NA>,False,60,60,rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR,KQkq,None,0,20,32,False
2,202312,202312092653864,3,d2d4,0,True,60,60,rnbqkbnr/pppp1ppp/4p3/8/4P3/8/PPPP1PPP/RNBQKBNR,KQkq,None,0,30,32,False
3,202312,202312092653864,4,d7d5,0,False,60,60,rnbqkbnr/pppp1ppp/4p3/8/3PP3/8/PPP2PPP/RNBQKBNR,KQkq,None,0,30,32,False
4,202312,202312092653864,5,c2c3,3,True,60,60,rnbqkbnr/ppp2ppp/4p3/3p4/3PP3/8/PPP2PPP/RNBQKBNR,KQkq,None,0,38,32,False
5,202312,202312092653864,6,f7f6,3,False,60,57,rnbqkbnr/ppp2ppp/4p3/3p4/3PP3/2P5/PP3PPP/RNBQKBNR,KQkq,None,0,34,32,False
6,202312,202312092653864,7,f2f3,0,True,57,57,rnbqkbnr/ppp3pp/4pp2/3p4/3PP3/2P5/PP3PPP/RNBQKBNR,KQkq,None,0,39,32,False
7,202312,202312092653864,8,c7c6,0,False,57,57,rnbqkbnr/ppp3pp/4pp2/3p4/3PP3/2P2P2/PP4PP/RNBQ...,KQkq,None,0,30,32,False
8,202312,202312092653864,9,g1e2,2,True,57,57,rnbqkbnr/pp4pp/2p1pp2/3p4/3PP3/2P2P2/PP4PP/RNB...,KQkq,None,0,35,32,False
9,202312,202312092653864,10,b7b6,0,False,57,55,rnbqkbnr/pp4pp/2p1pp2/3p4/3PP3/2P2P2/PP2N1PP/R...,KQkq,None,1,31,32,False


Note that the query above contains a seemingly unnecessary additional condition, i.e., `partition = 202312`. However, this additional information helps to narrow down and speed up the search, which is critical in this case because the `moves` table contains hundreds of billions of records.

In general, it is advised that queries on the `moves` table are formed with the combination of `partition` and `gid`, where the former is simply the first 6 digits of the latter.

Hence, we can retrieve and save moves from all games by the best player as follows:

In [29]:
# Initiate an empty table to store move-level information
conn.sql("""
CREATE OR REPLACE TABLE moves_in_games_by_best_player AS (
    SELECT
        *
    FROM
        core.moves
    WHERE
        1 = 0
);
""")

In [30]:
# Get IDs of all games by the best player
best_player_games = conn.sql("""
SELECT
    gid
FROM
    games_by_best_player;
""").df()

In [31]:
%%time

# Retrieve and save moves from all games by the best player
for gid in best_player_games["gid"]:
    partition = str(gid)[:6] # First 6 digits
    conn.sql(f"""
        INSERT INTO moves_in_games_by_best_player BY NAME (
            SELECT
                *
            FROM
                core.moves
            WHERE
                partition = {partition}
                AND gid = {gid}
        );
    """)

CPU times: user 18.4 s, sys: 5.61 s, total: 24.1 s
Wall time: 31 s


In [32]:
# Check result
conn.sql("""
SELECT
    *
FROM
    moves_in_games_by_best_player;
""").df()

,partition,gid,move_ply,move_uci,move_time,player_white,player_clock_time,opponent_clock_time,board_position,castling_rights,en_passant_targets,halfmove_clock,n_possible_moves,n_pieces,player_in_check
0,202312,202312092653864,1,e2e4,<NA>,True,60,60,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR,KQkq,None,0,20,32,False
1,202312,202312092653864,2,e7e6,<NA>,False,60,60,rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR,KQkq,None,0,20,32,False
2,202312,202312092653864,3,d2d4,0,True,60,60,rnbqkbnr/pppp1ppp/4p3/8/4P3/8/PPPP1PPP/RNBQKBNR,KQkq,None,0,30,32,False
3,202312,202312092653864,4,d7d5,0,False,60,60,rnbqkbnr/pppp1ppp/4p3/8/3PP3/8/PPP2PPP/RNBQKBNR,KQkq,None,0,30,32,False
4,202312,202312092653864,5,c2c3,3,True,60,60,rnbqkbnr/ppp2ppp/4p3/3p4/3PP3/8/PPP2PPP/RNBQKBNR,KQkq,None,0,38,32,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
517,202312,202312092741529,67,h5f7,0,True,20,40,4Q3/2R2pk1/pp3r1p/7B/4p3/P6P/1P3PP1/6K1,None,None,0,45,16,False
518,202312,202312092741529,68,f6f7,0,False,40,20,4Q3/2R2Bk1/pp3r1p/8/4p3/P6P/1P3PP1/6K1,None,None,0,14,15,False
519,202312,202312092741529,69,e8f7,1,True,20,40,4Q3/2R2rk1/pp5p/8/4p3/P6P/1P3PP1/6K1,None,None,0,39,14,False
520,202312,202312092741529,70,g7h8,1,False,40,19,8/2R2Qk1/pp5p/8/4p3/P6P/1P3PP1/6K1,None,None,0,1,13,True


If you need to retrieve moves for hundreds or thousands of games, you can further speed up the process by directly querying the source files, like so:

In [33]:
# Reset the table to store move-level information
conn.sql("""TRUNCATE moves_in_games_by_best_player;""")

In [34]:
%%time

# Retrieve and save moves from all games by the best player
for gid in best_player_games["gid"]:
    partition = str(gid)[:6] # First 6 digits
    segment = str(gid)[6:9]  # Next 3 digits
    conn.sql(f"""
        INSERT INTO moves_in_games_by_best_player BY NAME (
            SELECT
                *
            FROM
                '/scratch/gpfs/GRIFFITHS/chess-db/rawdata/partition={partition}/{segment}-moves.parquet'
            WHERE
                gid = {gid}
        );
    """)

CPU times: user 2.89 s, sys: 177 ms, total: 3.06 s
Wall time: 1.66 s


In [35]:
# Check all existing tables
conn.sql("""
SHOW ALL TABLES;
""").df()

,database,schema,name,column_names,column_types,temporary
0,core,main,games,"[gid, id, utc_datetime, initial_clock, clock_i...","[UBIGINT, VARCHAR, TIMESTAMP_S, USMALLINT, UTI...",False
1,core,main,moves,"[partition, gid, move_ply, move_uci, move_time...","[UINTEGER, UBIGINT, USMALLINT, VARCHAR, INTEGE...",False
2,core,main,players,"[id, first_game_datetime, last_game_datetime, ...","[VARCHAR, TIMESTAMP_S, TIMESTAMP_S, ENUM('GM',...",False
3,personal,main,games_by_best_player,"[gid, id, utc_datetime, initial_clock, clock_i...","[UBIGINT, VARCHAR, TIMESTAMP_S, USMALLINT, UTI...",False
4,personal,main,moves_in_games_by_best_player,"[partition, gid, move_ply, move_uci, move_time...","[UINTEGER, UBIGINT, USMALLINT, VARCHAR, INTEGE...",False
5,personal,main,top_players,"[id, first_game_datetime, last_game_datetime, ...","[VARCHAR, TIMESTAMP_S, TIMESTAMP_S, ENUM('GM',...",False


We confirm a more than 10x increase in speed.

## Remove tables

If we want, we can remove tables that we have created.

In [36]:
# Drop tables
conn.sql("""
DROP TABLE IF EXISTS moves_in_games_by_best_player;
DROP TABLE IF EXISTS games_by_best_player;
DROP TABLE IF EXISTS top_players;
""")

In [37]:
# Check all existing tables
conn.sql("""
SHOW ALL TABLES;
""").df()

,database,schema,name,column_names,column_types,temporary
0,core,main,games,"[gid, id, utc_datetime, initial_clock, clock_i...","[UBIGINT, VARCHAR, TIMESTAMP_S, USMALLINT, UTI...",False
1,core,main,moves,"[partition, gid, move_ply, move_uci, move_time...","[UINTEGER, UBIGINT, USMALLINT, VARCHAR, INTEGE...",False
2,core,main,players,"[id, first_game_datetime, last_game_datetime, ...","[VARCHAR, TIMESTAMP_S, TIMESTAMP_S, ENUM('GM',...",False


Attempting to remove tables in the primary database (aliased as `core`) will result in an error as we can only read from it.

## Closing DB Connection

To avoid database lock, ensure to properly close the connection at the end of each session.

In [38]:
# Close database connection
conn.close()

## Tips

The following are some general tips for effective use of the database:
- Always filter a desired subset first and then run queries on it.
- Order tables before joining them. Ordered join keys can save you a lot of time.
- Test a complex query on a small subset for resource estimation. Then, run the full query with the proper resource allocation.
- Deploy a Slurm job if the query requires long processing time and/or a lot of compute resources.
- Always close your connection at the end.